# Crop Insurance Pricing — Interactive Tool
Run the cell below. Fill the form and click **Run Pricing**.

In [ ]:
# ── Install missing packages if needed ──────────────────────────────────────
import importlib, subprocess, sys
for pkg in ['ipywidgets', 'openpyxl', 'scipy']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import datetime, os, warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Import the pricing engine (must be in same folder as this notebook) ──────
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
import crop_pricing_calculator_v3_final as _eng
from crop_pricing_calculator_v3_final import (
    calculate_all, calculate_losscost, calculate_bc,
    load_suminsured, calculate_cupncap, calculate_corridor,
    CORRIDOR_MODELS, SI_JOIN_KEYS
)
import pandas as pd
from openpyxl import Workbook

# ────────────────────────────────────────────────────────────────────────────
#  STYLES
# ────────────────────────────────────────────────────────────────────────────
display(HTML("""
<style>
.cip-wrap { font-family: Arial, sans-serif; max-width: 720px; }
.cip-header {
    display: flex; align-items: center; gap: 16px;
    border: 1px solid #ddd; border-radius: 8px;
    padding: 12px 20px; margin-bottom: 24px;
    background: #fff;
}
.cip-logo { font-size: 13px; font-weight: 700; color: #c0392b; line-height: 1.4; }
.cip-logo small { display:block; font-size:9px; font-weight:400;
                   color:#888; letter-spacing:.05em; }
.cip-title { font-size: 17px; font-weight: 600; color: #222;
              flex: 1; text-align: center; }
.cip-field {
    display: grid; grid-template-columns: 230px 1fr;
    align-items: center; gap: 12px; margin-bottom: 14px;
}
.cip-field label { font-size: 14px; color: #333; }
.cip-hint { font-size: 11px; color: #999; margin-top: 3px; }
.cip-section { font-size: 11px; font-weight: 600; color: #888;
                text-transform: uppercase; letter-spacing: .06em;
                margin: 20px 0 10px; border-top: 1px solid #eee; padding-top: 12px; }
.cip-btn {
    background: #185FA5 !important; color: #fff !important;
    border: none !important; border-radius: 6px !important;
    padding: 9px 28px !important; font-size: 14px !important;
    font-weight: 600 !important; cursor: pointer !important;
    margin-top: 16px;
}
.cip-ok  { background:#e8f5e9; border:1px solid #81c784;
            color:#2e7d32; border-radius:6px; padding:12px 16px;
            margin-top:16px; font-size:13px; }
.cip-err { background:#fdecea; border:1px solid #ef9a9a;
            color:#c62828; border-radius:6px; padding:12px 16px;
            margin-top:16px; font-size:13px; }
.cip-log { background:#f5f5f5; border:1px solid #ddd;
            border-radius:6px; padding:12px 16px;
            margin-top:16px; font-size:12px;
            font-family:monospace; white-space:pre-wrap; }
</style>
"""))

# ────────────────────────────────────────────────────────────────────────────
#  HEADER
# ────────────────────────────────────────────────────────────────────────────
display(HTML("""
<div class="cip-wrap">
  <div class="cip-header">
    <div class="cip-logo">IndusInd<br>
      <small>GENERAL INSURANCE</small>
      <small>FORMERLY RELIANCE GENERAL INSURANCE</small>
    </div>
    <div class="cip-title">Crop Insurance Pricing</div>
  </div>
  <div class="cip-section">Input files</div>
</div>
"""))

# ────────────────────────────────────────────────────────────────────────────
#  WIDGETS
# ────────────────────────────────────────────────────────────────────────────
style  = {'description_width': '230px'}
layout = widgets.Layout(width='420px')

w_input = widgets.Text(
    description='Historical yield file path:',
    placeholder=r'D:\Crop Insurance Pricing\input\Karnataka.xlsx',
    style=style, layout=layout
)
w_si = widgets.Text(
    description='Sum Insured (SI) file path:',
    placeholder=r'D:\Crop Insurance Pricing\input\Suminsured.xlsx',
    style=style, layout=layout
)
w_year = widgets.BoundedIntText(
    description='Base detrend year:',
    value=2024, min=1980, max=2099,
    style=style, layout=widgets.Layout(width='320px')
)
w_calc = widgets.SelectMultiple(
    description='Type of calculation:',
    options=[
        ('BC & T-Test + 60-130 + 80-110  (full run)', 'full'),
        ('BC & T-Test only (no corridor)',              'bc_ttest'),
        ('60-130 corridor only',                        '60_130'),
        ('80-110 corridor only',                        '80_110'),
    ],
    value=['full'],
    rows=4,
    style=style, layout=widgets.Layout(width='520px', height='110px')
)
w_output = widgets.Text(
    description='Output folder path:',
    placeholder=r'D:\Crop Insurance Pricing\output',
    style=style, layout=layout
)
w_btn = widgets.Button(
    description='▶  Run Pricing',
    button_style='',
    layout=widgets.Layout(width='180px', height='38px')
)
w_btn.add_class('cip-btn')
out = widgets.Output()

display(w_input)
display(HTML('<div class="cip-hint" style="margin-left:234px;">Accepts .xlsx or .csv</div>'))
display(w_si)
display(HTML('<div class="cip-hint" style="margin-left:234px;">Must have columns: Cluster, Season, State, District, Crop Name as per BOQ</div>'))
display(HTML('<div class="cip-section" style="max-width:720px">Configuration</div>'))
display(w_year)
display(HTML('<div class="cip-hint" style="margin-left:234px;">Should match DETREND_BASE_YEAR in your Excel formula</div>'))
display(w_calc)
display(HTML('<div class="cip-hint" style="margin-left:234px;">Hold Ctrl / ⌘ to select multiple</div>'))
display(HTML('<div class="cip-section" style="max-width:720px">Output</div>'))
display(w_output)
display(HTML('<div class="cip-hint" style="margin-left:234px;">Output is auto-named: &lt;input_name&gt;_output_YYYY-MM-DD_HHMMSS.xlsx</div>'))
display(w_btn)
display(out)

# ────────────────────────────────────────────────────────────────────────────
#  RUN HANDLER
# ────────────────────────────────────────────────────────────────────────────
def on_run(b):
    with out:
        clear_output(wait=True)

        # ── Validate inputs ────────────────────────────────────────────────
        errors = []
        inp   = w_input.value.strip()
        si    = w_si.value.strip()
        odir  = w_output.value.strip()
        calcs = list(w_calc.value)

        if not inp:
            errors.append('Historical yield file path is required.')
        elif not os.path.isfile(inp):
            errors.append(f'Historical yield file not found: {inp}')

        if not si:
            errors.append('Sum Insured file path is required.')
        elif not os.path.isfile(si):
            errors.append(f'SI file not found: {si}')

        if not odir:
            errors.append('Output folder path is required.')

        if not calcs:
            errors.append('Select at least one calculation type.')

        if errors:
            display(HTML('<div class="cip-err"><b>Please fix the following:</b><br>• ' +
                         '<br>• '.join(errors) + '</div>'))
            return

        # ── Determine which sheets to compute ──────────────────────────────
        run_full     = 'full'    in calcs
        run_bc       = run_full or 'bc_ttest' in calcs
        run_60_130   = run_full or '60_130'   in calcs
        run_80_110   = run_full or '80_110'   in calcs
        run_corridor = run_60_130 or run_80_110

        corridor_models = []
        if run_80_110: corridor_models.append((80, 110))
        if run_60_130: corridor_models.append((60, 130))

        # ── Override DETREND_BASE_YEAR in engine ───────────────────────────
        _eng.DETREND_BASE_YEAR = w_year.value

        logs = []
        def log(msg):
            logs.append(msg)
            print(msg)

        try:
            os.makedirs(odir, exist_ok=True)
            stem     = os.path.splitext(os.path.basename(inp))[0]
            ts       = datetime.datetime.now().strftime('%Y-%m-%d_%H%M%S')
            out_file = os.path.join(odir, f'{stem}_output_{ts}.xlsx')

            display(HTML('<div class="cip-log">Starting run...</div>'))

            log(f'Input      : {inp}')
            log(f'SI file    : {si}')
            log(f'Base year  : {w_year.value}')
            log(f'Output     : {out_file}')
            log('')

            # ── Read input ─────────────────────────────────────────────────
            df_in = pd.read_csv(inp) if inp.lower().endswith('.csv') else pd.read_excel(inp)
            log(f'Rows loaded: {len(df_in):,}')

            df_base = None
            df_lc   = None

            # ── Sheets 1-3: Base Data, LossCost, BC ───────────────────────
            if run_bc:
                log('[1/5] Computing Base Data & T-Test ...')
                df_base = calculate_all(df_in)
                log(f'      {len(df_base):,} rows x {len(df_base.columns)} cols')

                log('[2/5] Computing LossCost ...')
                df_lc = calculate_losscost(df_base)
                log(f'      {len(df_lc):,} rows x {len(df_lc.columns)} cols')

                log('[3/5] Computing BC ...')
                df_bc = calculate_bc(df_lc)
                log(f'      {len(df_bc):,} rows x {len(df_bc.columns)} cols')
            else:
                log('[1-3] Skipped (BC & T-Test not selected)')

            # ── Sheet 4: Suminsured ────────────────────────────────────────
            log('[4/5] Loading Suminsured ...')
            df_si_df = load_suminsured(si)
            si_cols  = [c for c in df_si_df.columns if c not in SI_JOIN_KEYS]
            log(f'      {len(df_si_df):,} rows | SI cols: {si_cols}')

            # ── Sheets 5+: CupnCap & Corridor (returns Workbook objects) ──
            cupncap_sheets = {}   # si_col -> DataFrame
            corridor_wbs   = {}   # sheet_name -> Workbook

            if run_corridor and df_lc is not None:
                log('[5/5] Computing CupnCap & Corridor sheets ...')
                for si_col in si_cols:
                    df_cc = calculate_cupncap(df_lc, df_si_df, si_col)
                    cupncap_sheets[si_col] = df_cc
                    log(f'      CupnCapModel_{si_col}: {len(df_cc):,} rows')
                    for floor_pct, cap_pct in corridor_models:
                        # calculate_corridor returns an openpyxl Workbook
                        wb_corr    = calculate_corridor(df_cc, si_col,
                                                        floor_pct / 100, cap_pct / 100)
                        sheet_name = f'{floor_pct}-{cap_pct} {si_col[:12]}'[:31]
                        corridor_wbs[sheet_name] = wb_corr
                        log(f'      {sheet_name}: done')
            elif run_corridor:
                log('[5/5] Corridor skipped — BC & T-Test must run first.')
            else:
                log('[5/5] Corridor not selected — skipped.')

            # ── Write all sheets to one Excel file ─────────────────────────
            log('')
            log('Writing output ...')
            with pd.ExcelWriter(out_file, engine='openpyxl') as writer:
                if df_base is not None:
                    df_base.to_excel(writer, index=False, sheet_name='Base Data & T Test')
                if df_lc is not None:
                    df_lc.to_excel(writer,   index=False, sheet_name='LossCost')
                    df_bc.to_excel(writer,   index=False, sheet_name='BC')
                df_si_df.to_excel(writer,    index=False, sheet_name='Suminsured')
                for si_col, df_cc in cupncap_sheets.items():
                    df_cc.to_excel(writer, index=False,
                                   sheet_name=f'CupnCapModel_{si_col}'[:31])
                # Corridor sheets: copy from their Workbook objects into main workbook
                main_wb = writer.book
                for sheet_name, wb_corr in corridor_wbs.items():
                    src_ws = wb_corr.active
                    new_ws = main_wb.create_sheet(title=sheet_name)
                    for row in src_ws.iter_rows():
                        for cell in row:
                            new_ws.cell(row=cell.row, column=cell.column,
                                        value=cell.value)
                    for col_letter, dim in src_ws.column_dimensions.items():
                        new_ws.column_dimensions[col_letter].width = dim.width

            written = []
            if df_base is not None: written += ['Base Data & T Test', 'LossCost', 'BC']
            written += ['Suminsured']
            written += [f'CupnCapModel_{c}'[:31] for c in cupncap_sheets]
            written += list(corridor_wbs.keys())
            log(f'Done!  Sheets: {written}')

            display(HTML(
                f'<div class="cip-ok">'
                f'<b>&#10004; Run complete!</b><br>'
                f'Output saved to:<br><code>{out_file}</code><br><br>'
                f'Sheets: ' + ', '.join(f'<b>{s}</b>' for s in written) +
                f'</div>'
            ))

        except Exception as e:
            import traceback
            display(HTML(
                f'<div class="cip-err"><b>Error:</b> {e}<br><br>'
                f'<pre style="font-size:11px">{traceback.format_exc()}</pre></div>'
            ))

w_btn.on_click(on_run)
